In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
import django

# Set up Django environment
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "agentcpq.settings")
django.setup()

# Import models after setting up Django
from cpq.models import Product

print("Django environment is ready!")

Django environment is ready!


In [13]:
from asgiref.sync import sync_to_async
import asyncio

# Wrap Django ORM queries in a sync function
def get_products():
    return list(Product.objects.all())

# Run inside an async-safe wrapper
products = asyncio.run(sync_to_async(get_products, thread_sensitive=True)())

# Print results
print(products)

[<Product: AI-Powered CPQ System>]


In [12]:
## THIS IS THE CREATE PRODUCT AGENT


In [8]:

import os
import django
import nest_asyncio
import asyncio
from asgiref.sync import sync_to_async

# 🔹 Patch Jupyter to allow Django ORM calls
nest_asyncio.apply()

# 🔹 Set up Django environment
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "agentcpq.settings")
django.setup()

from cpq.models import Product

class ProductCreatorAgent:
    """
    AI Agent responsible for creating products in AgentCPQ.
    """

    def __init__(self):
        print("✅ Product Creator Agent Initialized")

    def validate_product_data(self, data):
        """Ensure required fields are present and valid."""
        required_fields = ["name", "sku", "price"]
        missing_fields = [field for field in required_fields if field not in data]

        if missing_fields:
            return False, f"⚠️ Missing required fields: {', '.join(missing_fields)}"

        try:
            data["price"] = float(data["price"])  # Ensure price is a valid float
        except ValueError:
            return False, "⚠️ Price must be a valid number"

        return True, data

    async def create_product(self, data):
        """Create a product record in the database (Async Safe)."""
        # Validate data
        is_valid, result = self.validate_product_data(data)
        if not is_valid:
            return {"success": False, "message": result}

        # ✅ Wrap Django ORM call inside `sync_to_async`
        product = await sync_to_async(Product.objects.create, thread_sensitive=True)(
            name=result["name"],
            sku=result["sku"],
            price=result["price"],
            is_subscription=result.get("is_subscription", False),
            term=result.get("term"),
            is_bundle=result.get("is_bundle", False),
        )

        return {
            "success": True,
            "message": f"✅ Product '{product.name}' created successfully!",
            "product_id": product.id
        }

# ✅ Initialize Agent
agent = ProductCreatorAgent()
print("🚀 Agent Ready to Create Products!")

✅ Product Creator Agent Initialized
🚀 Agent Ready to Create Products!


In [9]:
# Example product data
product_data = {
    "name": "AI-Powered CPQ System",
    "sku": "AI-CPQ-001",
    "price": "59.99",
    "is_subscription": True,
    "term": 12,
}

# ✅ Run the function properly inside Jupyter
response = asyncio.run(agent.create_product(product_data))
print(response)

{'success': True, 'message': "✅ Product 'AI-Powered CPQ System' created successfully!", 'product_id': 77}
